# FrozenLake LVR training on Colab A100

This notebook trains the custom FrozenLake path for **Qwen2.5-VL-3B-Instruct** on one A100 40 GB. It uses LoRA for the language path and directly trains the task-specific latent-end vector; the vision tower remains frozen.

The model sees only the initial board image and the game instruction. Intermediate successor-state images supervise latent tokens. The textual target contains only the navigation actions.

This Colab path deliberately uses PyTorch SDPA. It does not install FlashAttention, avoiding binary compatibility failures between Colab's PyTorch runtime and cached FlashAttention wheels.

Safety gates are deliberate: environment validation and a one-batch forward pass must succeed before pilot or full training. Pilot, full training, validation, and final test are opt-in cells.

## Run order

1. Run **Configuration**, **Hardware check**, **Clone**, and **Install**.
2. The install cell restarts the runtime. After reconnection, continue at **Restore configuration**; do not rerun the install cell.
3. Download, verify, extract, validate, preflight, and smoke-test the dataset/model.
4. Enable the 10-step pilot only after the smoke test reports `status: ok`.
5. Enable full training only after the pilot completes successfully.

The full job writes to Google Drive by default for persistence. This is slower than local Colab storage, but survives runtime recycling.

In [ ]:
# @title 1. Configuration
from pathlib import Path
import json

REPO_URL = "https://github.com/Noridom1/lvr.git"  # @param {type:"string"}
REVISION = "main"  # @param {type:"string"}
DRIVE_URL = "https://drive.google.com/file/d/1XhmzxEM0HCn6CTnNVkR7cCQEHaXujZqA/view?usp=sharing"  # @param {type:"string"}
ARCHIVE_SHA256 = "D7B3C86A41F3CB1BE5468E9D7130E5F7568BC1CECB47DD93380A16392F705540"
EXPECTED_ARCHIVE_BYTES = 1842169309
MODEL_NAME = "Qwen/Qwen2.5-VL-3B-Instruct"  # @param {type:"string"}

CONFIG_PATH = Path("/content/frozenlake_lvr_colab_config.json")
REPO_DIR = Path("/content/lvr")
ARCHIVE_PATH = Path("/content/frozenlake_training_samples.zip")
DATA_ROOT = REPO_DIR / "training_samples" / "frozenlake"

config = {
    "repo_url": REPO_URL,
    "revision": REVISION,
    "drive_url": DRIVE_URL,
    "archive_sha256": ARCHIVE_SHA256,
    "expected_archive_bytes": EXPECTED_ARCHIVE_BYTES,
    "model_name": MODEL_NAME,
}
CONFIG_PATH.write_text(json.dumps(config, indent=2), encoding="utf-8")
print(json.dumps(config, indent=2))
print(f"Configuration saved across runtime restart: {CONFIG_PATH}")

In [ ]:
# @title 2. Check the assigned Colab hardware
import shutil
import subprocess
import psutil
import torch

subprocess.run(["nvidia-smi"], check=True)

if not torch.cuda.is_available():
    raise RuntimeError("CUDA is unavailable. Select an A100 GPU runtime before continuing.")

gpu_name = torch.cuda.get_device_name(0)
gpu_gib = torch.cuda.get_device_properties(0).total_memory / 1024**3
memory = psutil.virtual_memory()
disk = shutil.disk_usage("/content")

print(f"GPU: {gpu_name} ({gpu_gib:.1f} GiB)")
print(f"RAM: {memory.total / 1024**3:.1f} GiB total, {memory.available / 1024**3:.1f} GiB free")
print(f"Disk: {disk.free / 1024**3:.1f} GiB free")

if gpu_gib < 39:
    raise RuntimeError("This configuration expects approximately 40 GiB GPU VRAM.")
if memory.total / 1024**3 < 24:
    raise RuntimeError("This LoRA configuration expects at least approximately 24 GiB system RAM.")
if disk.free / 1024**3 < 50:
    raise RuntimeError("At least approximately 50 GiB free local disk is recommended.")

In [ ]:
# @title 3. Clone the repository and resolve the requested revision
import subprocess
from pathlib import Path

if (REPO_DIR / ".git").is_dir():
    print(f"Reusing existing clone: {REPO_DIR}")
else:
    if REPO_DIR.exists() and any(REPO_DIR.iterdir()):
        raise RuntimeError(f"{REPO_DIR} exists and is not an empty Git checkout.")
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)

subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "origin"], check=True)
if REVISION == "main":
    subprocess.run(["git", "-C", str(REPO_DIR), "checkout", "main"], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only", "origin", "main"], check=True)
else:
    subprocess.run(["git", "-C", str(REPO_DIR), "checkout", "--detach", REVISION], check=True)

resolved_revision = subprocess.check_output(
    ["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"],
    text=True,
).strip()
print(f"Resolved revision: {resolved_revision}")

## Install boundary

The next cell removes any preinstalled FlashAttention wheel, installs the official PyTorch 2.6.0 CUDA 12.4 wheels, then installs only the dependencies imported by the FrozenLake path. It restarts the Colab runtime so the new PyTorch libraries load cleanly.

This training configuration uses PyTorch SDPA throughout.

In [ ]:
# @title 4. Install FrozenLake dependencies and restart
import importlib.util
from importlib.metadata import PackageNotFoundError, version
import os
import signal
import subprocess
import sys

# A FlashAttention wheel compiled for another PyTorch ABI can break the Qwen
# import even when training asks for SDPA. Remove it before replacing PyTorch.
try:
    installed_flash_version = version("flash-attn")
except PackageNotFoundError:
    print("FlashAttention is not installed; SDPA setup is already clean.")
else:
    print(f"Removing FlashAttention {installed_flash_version} before installing PyTorch.")
    subprocess.run(
        [sys.executable, "-m", "pip", "uninstall", "-y", "flash-attn"],
        check=True,
    )

PYTORCH_INDEX = "https://download.pytorch.org/whl/cu124"
subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "torch==2.6.0",
        "torchvision==0.21.0",
        "torchaudio==2.6.0",
        "--index-url",
        PYTORCH_INDEX,
    ],
    check=True,
)
subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-r",
        str(REPO_DIR / "requirements-frozenlake-colab.txt"),
    ],
    check=True,
)

if importlib.util.find_spec("google.colab") is not None:
    print("Dependencies installed. Restarting the Colab Python process now.")
    os.kill(os.getpid(), signal.SIGKILL)
else:
    print("Dependencies installed. Restart the Python kernel manually, then continue below.")

## Continue here after the runtime reconnects

The configuration file under `/content` survives a normal Colab runtime restart. Run the next cell to restore all variables.

In [ ]:
# @title 5. Restore configuration after restart
from importlib.metadata import PackageNotFoundError, version
from pathlib import Path
import json
import os
import subprocess
import sys

CONFIG_PATH = Path("/content/frozenlake_lvr_colab_config.json")
if not CONFIG_PATH.is_file():
    raise RuntimeError("Configuration is missing. Run the Configuration cell first.")

config = json.loads(CONFIG_PATH.read_text(encoding="utf-8"))
REPO_URL = config["repo_url"]
REVISION = config["revision"]
DRIVE_URL = config["drive_url"]
ARCHIVE_SHA256 = config["archive_sha256"].upper()
EXPECTED_ARCHIVE_BYTES = int(config["expected_archive_bytes"])
MODEL_NAME = config["model_name"]

REPO_DIR = Path("/content/lvr")
ARCHIVE_PATH = Path("/content/frozenlake_training_samples.zip")
DATA_ROOT = REPO_DIR / "training_samples" / "frozenlake"
if not (REPO_DIR / ".git").is_dir():
    raise RuntimeError("Repository clone is missing. Run the clone cell before installing.")

os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
from scripts.colab_subprocess import run_streaming

# Keep this check before importing torch or transformers. If a dependency ever
# brings FlashAttention back, remove it before Transformers can auto-detect it.
try:
    unexpected_flash_version = version("flash-attn")
except PackageNotFoundError:
    unexpected_flash_version = None
if unexpected_flash_version is not None:
    print(f"Removing unexpected FlashAttention {unexpected_flash_version}.")
    run_streaming(
        [sys.executable, "-m", "pip", "uninstall", "-y", "flash-attn"]
    )

ATTENTION = "sdpa"
DISABLE_FLASH_ATTN2 = "True"
os.environ["DISABLE_FLASH_ATTN2"] = DISABLE_FLASH_ATTN2

import torch
import transformers
if torch.__version__.split("+")[0] != "2.6.0":
    raise RuntimeError(f"Expected torch 2.6.0 after restart; found {torch.__version__}.")
print(f"Repository: {REPO_DIR}")
print(f"Torch: {torch.__version__}")
print(f"Transformers: {transformers.__version__}")
print("FlashAttention: disabled (PyTorch ABI-safe Colab setup)")
print(f"Attention backend: {ATTENTION}")

In [ ]:
# @title 6. Download the FrozenLake ZIP and verify its exact checksum
import hashlib
from pathlib import Path
import gdown

def sha256_file(path: Path, chunk_size: int = 8 * 1024 * 1024) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for chunk in iter(lambda: stream.read(chunk_size), b""):
            digest.update(chunk)
    return digest.hexdigest().upper()

if ARCHIVE_PATH.is_file():
    existing_hash = sha256_file(ARCHIVE_PATH)
    if ARCHIVE_PATH.stat().st_size != EXPECTED_ARCHIVE_BYTES or existing_hash != ARCHIVE_SHA256:
        raise RuntimeError(
            f"Existing archive failed verification: {ARCHIVE_PATH}. "
            "Move it aside and rerun this cell."
        )
    print("Verified existing archive; download skipped.")
else:
    partial_path = ARCHIVE_PATH.with_suffix(".zip.part")
    downloaded = gdown.download(
        url=DRIVE_URL,
        output=str(partial_path),
        quiet=False,
        fuzzy=True,
        resume=True,
    )
    if not downloaded or not partial_path.is_file():
        raise RuntimeError("Google Drive download did not produce the expected file.")

    actual_bytes = partial_path.stat().st_size
    actual_hash = sha256_file(partial_path)
    print(f"Downloaded bytes: {actual_bytes}")
    print(f"SHA-256: {actual_hash}")
    if actual_bytes != EXPECTED_ARCHIVE_BYTES:
        raise RuntimeError(f"Archive size mismatch: expected {EXPECTED_ARCHIVE_BYTES}, got {actual_bytes}.")
    if actual_hash != ARCHIVE_SHA256:
        raise RuntimeError(f"Archive checksum mismatch: expected {ARCHIVE_SHA256}, got {actual_hash}.")
    partial_path.replace(ARCHIVE_PATH)

print(f"Archive ready: {ARCHIVE_PATH} ({ARCHIVE_PATH.stat().st_size / 1024**3:.3f} GiB)")

In [ ]:
# @title 7. Safely extract into training_samples/frozenlake
import os
from pathlib import Path
from zipfile import ZipFile

EXPECTED_PNGS = 16806
EXPECTED_TRACES = 4000
data_parent = REPO_DIR / "training_samples"

def extracted_counts() -> tuple[int, int]:
    if not DATA_ROOT.is_dir():
        return 0, 0
    return (
        sum(1 for _ in DATA_ROOT.glob("[0-9]" * 8 + "/frame_*.png")),
        sum(1 for _ in DATA_ROOT.glob("[0-9]" * 8 + "/trace.json")),
    )

png_count, trace_count = extracted_counts()
if (png_count, trace_count) == (EXPECTED_PNGS, EXPECTED_TRACES):
    print("Dataset is already fully extracted; extraction skipped.")
else:
    data_parent.mkdir(parents=True, exist_ok=True)
    destination = data_parent.resolve()
    with ZipFile(ARCHIVE_PATH) as archive:
        infos = archive.infolist()
        files = [info.filename for info in infos if not info.is_dir()]
        png_entries = [
            name for name in files
            if name.startswith("frozenlake/") and name.endswith(".png")
        ]
        trace_entries = [
            name for name in files
            if name.startswith("frozenlake/") and name.endswith("/trace.json")
        ]
        unexpected = [
            name for name in files
            if not (
                name.startswith("frozenlake/")
                and (name.endswith(".png") or name.endswith("/trace.json"))
            )
        ]
        if len(png_entries) != EXPECTED_PNGS:
            raise RuntimeError(f"ZIP PNG count mismatch: {len(png_entries)}")
        if len(trace_entries) != EXPECTED_TRACES:
            raise RuntimeError(f"ZIP trace count mismatch: {len(trace_entries)}")
        if unexpected:
            raise RuntimeError(f"Unexpected ZIP entries: {unexpected[:10]}")

        for info in infos:
            target = (destination / info.filename).resolve()
            if target != destination and destination not in target.parents:
                raise RuntimeError(f"Unsafe path in ZIP: {info.filename}")
        archive.extractall(destination)

    png_count, trace_count = extracted_counts()

if (png_count, trace_count) != (EXPECTED_PNGS, EXPECTED_TRACES):
    raise RuntimeError(
        f"Extracted dataset mismatch: {png_count} PNGs, {trace_count} traces."
    )
print(f"Dataset ready: {DATA_ROOT}")
print(f"PNGs: {png_count}; traces: {trace_count}")

In [ ]:
# @title 8. Validate all manifests, paths, trajectories, and split isolation
import sys

run_streaming(
    [
        sys.executable,
        "scripts/validate_frozenlake_manifests.py",
        "--data-dir",
        "data/frozenlake",
        "--image-folder",
        str(DATA_ROOT),
    ],
    cwd=REPO_DIR,
)

In [ ]:
# @title 9. Run the Colab environment preflight
import sys

run_streaming(
    [
        sys.executable,
        "scripts/check_frozenlake_colab.py",
        "--attention",
        ATTENTION,
        "--workspace",
        str(REPO_DIR),
    ],
    cwd=REPO_DIR,
)
print("FrozenLake preflight passed.")

In [ ]:
# @title 10. One-batch GPU forward smoke test
import sys

run_streaming(
    [
        sys.executable,
        "-m",
        "scripts.smoke_test_frozenlake_lvr",
        "--model",
        MODEL_NAME,
        "--data-path",
        "data/frozenlake/train.jsonl",
        "--image-folder",
        str(DATA_ROOT),
        "--attention",
        ATTENTION,
    ],
    cwd=REPO_DIR,
)
print("Do not continue unless the smoke test reported status: ok and finite losses.")

## Ten-step optimizer pilot

Enable this only after the one-batch smoke test succeeds. On a single Colab A100 this uses rank-32 LoRA for the language path, keeps the vision tower frozen, and trains the latent-end vector directly. ZeRO-2 keeps the small optimizer on GPU and avoids the host-RAM spike caused by full-parameter CPU AdamW. The three-loss objective is unchanged.

In [ ]:
# @title 11. Run the 10-step pilot
RUN_PILOT = False  # @param {type:"boolean"}
PILOT_OUTPUT_DIR = "/content/frozenlake_pilot_lora"  # @param {type:"string"}

import os
from pathlib import Path

if not RUN_PILOT:
    print("Pilot disabled. Set RUN_PILOT=True after the smoke test passes.")
else:
    pilot_dir = Path(PILOT_OUTPUT_DIR)
    if pilot_dir.exists() and any(pilot_dir.iterdir()):
        raise RuntimeError(
            f"{pilot_dir} is not empty. Choose a new PILOT_OUTPUT_DIR to protect prior results."
        )
    env = os.environ.copy()
    env.update({
        "MODEL_NAME": MODEL_NAME,
        "DATA_PATH": "data/frozenlake/train.jsonl",
        "EVAL_DATA_PATH": "data/frozenlake/validation.jsonl",
        "IMAGE_FOLDER": str(DATA_ROOT),
        "OUTPUT_DIR": str(pilot_dir),
        "MAX_STEPS": "10",
        "GRADIENT_ACCUMULATION_STEPS": "2",
        "SAVE_STEPS": "1000",
        "EVAL_STEPS": "1000",
        "LORA_ENABLE": "True",
        "LORA_RANK": "32",
        "LORA_ALPHA": "64",
        "FREEZE_LLM": "True",
        "DEEPSPEED_CONFIG": "scripts/zero2.json",
        "DISABLE_FLASH_ATTN2": DISABLE_FLASH_ATTN2,
    })
    run_streaming(
        ["bash", "scripts/finetune_lvr_frozenlake_3b_colab.sh"],
        cwd=REPO_DIR,
        env=env,
    )

## Full training

This cell mounts Google Drive and uses a persistent output directory. Direct Drive checkpoint writes can be slower than local disk, but the retained checkpoint survives a Colab runtime reset.

For a fresh run, leave `CHECKPOINT_NAME` empty and ensure the output directory is empty. To resume, set it to the exact `checkpoint-N` directory inside the same output directory.

In [ ]:
# @title 12. Run or resume full training
RUN_FULL_TRAINING = False  # @param {type:"boolean"}
DRIVE_OUTPUT_DIR = "/content/drive/MyDrive/lvr_frozenlake/qwen2.5-vl-3b-lora"  # @param {type:"string"}
CHECKPOINT_NAME = ""  # @param {type:"string"}
NUM_TRAIN_EPOCHS = 5  # @param {type:"integer"}

import os
from pathlib import Path

if not RUN_FULL_TRAINING:
    print("Full training disabled. Enable it only after the 10-step pilot succeeds.")
else:
    from google.colab import drive
    drive.mount("/content/drive")

    output_dir = Path(DRIVE_OUTPUT_DIR)
    if output_dir.exists() and any(output_dir.iterdir()) and not CHECKPOINT_NAME:
        raise RuntimeError(
            f"{output_dir} is not empty. Set CHECKPOINT_NAME to resume or choose a new output directory."
        )
    output_dir.mkdir(parents=True, exist_ok=True)

    env = os.environ.copy()
    env.update({
        "MODEL_NAME": MODEL_NAME,
        "DATA_PATH": "data/frozenlake/train.jsonl",
        "EVAL_DATA_PATH": "data/frozenlake/validation.jsonl",
        "IMAGE_FOLDER": str(DATA_ROOT),
        "OUTPUT_DIR": str(output_dir),
        "CHECKPOINT_NAME": CHECKPOINT_NAME,
        "NUM_TRAIN_EPOCHS": str(NUM_TRAIN_EPOCHS),
        "MAX_STEPS": "-1",
        "GRADIENT_ACCUMULATION_STEPS": "16",
        "LORA_ENABLE": "True",
        "LORA_RANK": "32",
        "LORA_ALPHA": "64",
        "FREEZE_LLM": "True",
        "DEEPSPEED_CONFIG": "scripts/zero2.json",
        "DISABLE_FLASH_ATTN2": DISABLE_FLASH_ATTN2,
    })
    run_streaming(
        ["bash", "scripts/finetune_lvr_frozenlake_3b_colab.sh"],
        cwd=REPO_DIR,
        env=env,
    )

In [ ]:
# @title 13. Inspect current GPU, RAM, disk, and checkpoints
import shutil
import subprocess
import psutil
from pathlib import Path

subprocess.run(["nvidia-smi"], check=True)
memory = psutil.virtual_memory()
disk = shutil.disk_usage("/content")
print(f"RAM used: {(memory.total - memory.available) / 1024**3:.1f} GiB")
print(f"RAM free: {memory.available / 1024**3:.1f} GiB")
print(f"Local disk free: {disk.free / 1024**3:.1f} GiB")

output_dir = Path(DRIVE_OUTPUT_DIR)
if output_dir.exists():
    checkpoints = sorted(
        output_dir.glob("checkpoint-*"),
        key=lambda path: int(path.name.split("-")[-1]),
    )
    print("Checkpoints:", [str(path) for path in checkpoints])

## Evaluation gates

Use validation while selecting the latent-end threshold. Run the final test split only once after the configuration is fixed.

In [ ]:
# @title 14. Validation preview
RUN_VALIDATION = False  # @param {type:"boolean"}
EVALUATION_CHECKPOINT = "/content/drive/MyDrive/lvr_frozenlake/qwen2.5-vl-3b-lora"  # @param {type:"string"}
VALIDATION_MAX_SAMPLES = 20  # @param {type:"integer"}
LVR_END_THRESHOLD = 0.02  # @param {type:"number"}

import sys
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")
checkpoint_path = Path(EVALUATION_CHECKPOINT)
if not (checkpoint_path / "adapter_config.json").is_file():
    raise FileNotFoundError(
        f"Final LoRA adapter not found at {checkpoint_path}. "
        "Use the completed output directory, not an incomplete checkpoint."
    )

if not RUN_VALIDATION:
    print(f"Checkpoint ready: {checkpoint_path}")
    print("Validation disabled. Enable it to tune LVR_END_THRESHOLD.")
else:
    command = [
        sys.executable,
        "-m",
        "evaluation.evaluate_frozenlake_lvr",
        "--checkpoint",
        EVALUATION_CHECKPOINT,
        "--data-path",
        "data/frozenlake/validation.jsonl",
        "--image-folder",
        str(DATA_ROOT),
        "--output-dir",
        str(checkpoint_path / "validation_evaluation"),
        "--lvr-end-threshold",
        str(LVR_END_THRESHOLD),
        "--attention",
        ATTENTION,
    ]
    if VALIDATION_MAX_SAMPLES > 0:
        command.extend(["--max-samples", str(VALIDATION_MAX_SAMPLES)])
    run_streaming(command, cwd=REPO_DIR)

In [ ]:
# @title 15. Final test evaluation
RUN_FINAL_TEST = False  # @param {type:"boolean"}

import sys
from pathlib import Path

if not RUN_FINAL_TEST:
    print("Final test disabled. Enable it only after threshold selection is finished.")
else:
    run_streaming(
        [
            sys.executable,
            "-m",
            "evaluation.evaluate_frozenlake_lvr",
            "--checkpoint",
            EVALUATION_CHECKPOINT,
            "--data-path",
            "data/frozenlake/test.jsonl",
            "--image-folder",
            str(DATA_ROOT),
            "--output-dir",
            str(Path(EVALUATION_CHECKPOINT) / "test_evaluation"),
            "--lvr-end-threshold",
            str(LVR_END_THRESHOLD),
            "--attention",
            ATTENTION,
        ],
        cwd=REPO_DIR,
    )

In [ ]:
# @title 16. Inference on one test sample
RUN_SINGLE_SAMPLE = False  # @param {type:"boolean"}
EVALUATION_CHECKPOINT = "/content/drive/MyDrive/lvr_frozenlake/qwen2.5-vl-3b-lora"  # @param {type:"string"}
TEST_SAMPLE_INDEX = 0  # @param {type:"integer"}
LVR_END_THRESHOLD = 0.02  # @param {type:"number"}

import json
import sys
from IPython.display import Image, display
from pathlib import Path
from google.colab import drive

if not RUN_SINGLE_SAMPLE:
    print("Single-sample inference disabled. Choose an index from 0 to 199 and enable it.")
else:
    drive.mount("/content/drive")
    checkpoint_path = Path(EVALUATION_CHECKPOINT)
    if not (checkpoint_path / "adapter_config.json").is_file():
        raise FileNotFoundError(
            f"Final LoRA adapter not found at {checkpoint_path}. "
            "Use the completed output directory, not an incomplete checkpoint."
        )
    if not (checkpoint_path / "non_lora_state_dict.bin").is_file():
        raise FileNotFoundError(f"Latent-end weights are missing from {checkpoint_path}.")

    test_path = REPO_DIR / "data/frozenlake/test.jsonl"
    test_records = [
        json.loads(line)
        for line in test_path.read_text(encoding="utf-8").splitlines()
        if line.strip()
    ]
    if not 0 <= TEST_SAMPLE_INDEX < len(test_records):
        raise IndexError(
            f"TEST_SAMPLE_INDEX must be between 0 and {len(test_records) - 1}."
        )

    record = test_records[TEST_SAMPLE_INDEX]
    sample_output_dir = Path("/content/frozenlake_single_sample") / f"sample-{TEST_SAMPLE_INDEX:03d}"
    run_streaming(
        [
            sys.executable,
            "-m",
            "evaluation.evaluate_frozenlake_lvr",
            "--checkpoint",
            str(checkpoint_path),
            "--data-path",
            "data/frozenlake/test.jsonl",
            "--image-folder",
            str(DATA_ROOT),
            "--output-dir",
            str(sample_output_dir),
            "--sample-index",
            str(TEST_SAMPLE_INDEX),
            "--lvr-end-threshold",
            str(LVR_END_THRESHOLD),
            "--save-distance-trace",
            "--attention",
            ATTENTION,
        ],
        cwd=REPO_DIR,
    )

    prediction_lines = (sample_output_dir / "predictions.jsonl").read_text(encoding="utf-8").splitlines()
    prediction = json.loads(prediction_lines[0])
    image_path = Path(record["initial_image"])
    if not image_path.is_absolute():
        image_path = DATA_ROOT / image_path
    display(Image(filename=str(image_path)))
    print(f"Sample index: {TEST_SAMPLE_INDEX}")
    print(f"Sample id: {record['id']}")
    print(f"Instruction: {record['instruction']}")
    print("Expected actions:", " ".join(prediction["expected_actions"]))
    print("Predicted actions:", " ".join(prediction["predicted_actions"]))
    print(f"Latent exit: {prediction['latent_exit_reason']} after {prediction['latent_steps']} steps")
    print(f"Latent-end distance: min={prediction['latent_end_distance_min']}, final={prediction['latent_end_distance_final']}, threshold={prediction['lvr_end_threshold']}")
    print(f"Transition token: {prediction['transition_token']!r}; is <|lvr_end|>: {prediction['transition_is_lvr_end']}")
    print(f"Valid format: {prediction['valid_format']}")
    print(f"Exact match: {prediction['exact_match']}")
    print(f"Reached goal: {prediction['goal_success']} ({prediction['terminal_reason']})")
    print("Action-side model output:", prediction["action_output"])

In [ ]:
# @title 17. Diagnose one training sample
RUN_TRAIN_SAMPLE = False  # @param {type:"boolean"}
EVALUATION_CHECKPOINT = "/content/drive/MyDrive/lvr_frozenlake/qwen2.5-vl-3b-lora"  # @param {type:"string"}
TRAIN_SAMPLE_INDEX = 0  # @param {type:"integer"}
LVR_END_THRESHOLD = 0.02  # @param {type:"number"}
MAX_LVR_STEPS = 2048  # @param {type:"integer"}

import json
import sys
from IPython.display import Image, display
from pathlib import Path
from google.colab import drive

if not RUN_TRAIN_SAMPLE:
    print("Training-sample diagnosis disabled. Choose an index from 0 to 3599 and enable it.")
else:
    drive.mount("/content/drive")
    checkpoint_path = Path(EVALUATION_CHECKPOINT)
    if not (checkpoint_path / "adapter_config.json").is_file():
        raise FileNotFoundError(f"Final LoRA adapter not found at {checkpoint_path}.")
    if not (checkpoint_path / "non_lora_state_dict.bin").is_file():
        raise FileNotFoundError(f"Latent-end weights are missing from {checkpoint_path}.")

    train_path = REPO_DIR / "data/frozenlake/train.jsonl"
    train_records = [
        json.loads(line)
        for line in train_path.read_text(encoding="utf-8").splitlines()
        if line.strip()
    ]
    if not 0 <= TRAIN_SAMPLE_INDEX < len(train_records):
        raise IndexError(
            f"TRAIN_SAMPLE_INDEX must be between 0 and {len(train_records) - 1}."
        )

    record = train_records[TRAIN_SAMPLE_INDEX]
    sample_output_dir = checkpoint_path / "train_sample_diagnostics" / f"sample-{TRAIN_SAMPLE_INDEX:04d}"
    run_streaming(
        [
            sys.executable,
            "-m",
            "evaluation.evaluate_frozenlake_lvr",
            "--checkpoint",
            str(checkpoint_path),
            "--data-path",
            "data/frozenlake/train.jsonl",
            "--image-folder",
            str(DATA_ROOT),
            "--output-dir",
            str(sample_output_dir),
            "--sample-index",
            str(TRAIN_SAMPLE_INDEX),
            "--max-lvr-steps",
            str(MAX_LVR_STEPS),
            "--lvr-end-threshold",
            str(LVR_END_THRESHOLD),
            "--save-distance-trace",
            "--attention",
            ATTENTION,
        ],
        cwd=REPO_DIR,
    )

    prediction = json.loads(
        (sample_output_dir / "predictions.jsonl").read_text(encoding="utf-8").splitlines()[0]
    )
    initial_path = Path(record["initial_image"])
    if not initial_path.is_absolute():
        initial_path = DATA_ROOT / initial_path
    print("Initial frame:")
    display(Image(filename=str(initial_path)))
    print(f"Train index: {TRAIN_SAMPLE_INDEX}; id: {record['id']}")
    print("Ground-truth actions:", " ".join(prediction["expected_actions"]))
    print("Predicted actions:", " ".join(prediction["predicted_actions"]) or "<none>")
    print(f"Latent exit: {prediction['latent_exit_reason']} after {prediction['latent_steps']} steps")
    print(f"Distance first/min/final: {prediction['latent_end_distance_first']} / {prediction['latent_end_distance_min']} / {prediction['latent_end_distance_final']}")
    print(f"Threshold: {prediction['lvr_end_threshold']}")
    print(f"Transition token: {prediction['transition_token']!r}; is <|lvr_end|>: {prediction['transition_is_lvr_end']}")
    print("Action-side model output:", prediction["action_output"])
    distance_trace = prediction.get("latent_end_distance_trace", [])
    print("First 10 distances:", distance_trace[:10])
    print("Last 10 distances:", distance_trace[-10:])
    print(f"Saved diagnostics to: {sample_output_dir}")

## Fixed-length latent decoding

This inference path ignores the learned mode-switch threshold. It rolls out exactly 259 recurrent latent tokens (the training-set average), forces the `<|lvr_latent_end|>` boundary token, and then resumes ordinary action-token decoding.

In [ ]:
# @title 18. Fixed 259-token decoding on one train or test sample
RUN_FIXED_SAMPLE = False  # @param {type:"boolean"}
FIXED_SAMPLE_SPLIT = "test"  # @param ["test", "train"]
FIXED_SAMPLE_INDEX = 0  # @param {type:"integer"}
FIXED_LVR_STEPS = 259  # @param {type:"integer"}
EVALUATION_CHECKPOINT = "/content/drive/MyDrive/lvr_frozenlake/qwen2.5-vl-3b-lora"  # @param {type:"string"}

import json
import sys
from IPython.display import Image, display
from pathlib import Path
from google.colab import drive

if not RUN_FIXED_SAMPLE:
    print("Fixed-length sample inference disabled. Select train/test and enable it.")
else:
    if FIXED_SAMPLE_SPLIT not in {"train", "test"}:
        raise ValueError("FIXED_SAMPLE_SPLIT must be 'train' or 'test'.")
    if FIXED_LVR_STEPS <= 0:
        raise ValueError("FIXED_LVR_STEPS must be positive.")

    drive.mount("/content/drive")
    checkpoint_path = Path(EVALUATION_CHECKPOINT)
    if not (checkpoint_path / "adapter_config.json").is_file():
        raise FileNotFoundError(f"Final LoRA adapter not found at {checkpoint_path}.")
    if not (checkpoint_path / "non_lora_state_dict.bin").is_file():
        raise FileNotFoundError(f"Latent trainables are missing from {checkpoint_path}.")

    data_path = REPO_DIR / f"data/frozenlake/{FIXED_SAMPLE_SPLIT}.jsonl"
    records = [
        json.loads(line)
        for line in data_path.read_text(encoding="utf-8").splitlines()
        if line.strip()
    ]
    if not 0 <= FIXED_SAMPLE_INDEX < len(records):
        raise IndexError(
            f"FIXED_SAMPLE_INDEX must be between 0 and {len(records) - 1}."
        )

    record = records[FIXED_SAMPLE_INDEX]
    sample_output_dir = (
        Path("/content/frozenlake_fixed_samples")
        / FIXED_SAMPLE_SPLIT
        / f"sample-{FIXED_SAMPLE_INDEX:04d}"
    )
    run_streaming(
        [
            sys.executable,
            "-m",
            "evaluation.evaluate_frozenlake_lvr",
            "--checkpoint",
            str(checkpoint_path),
            "--data-path",
            str(data_path),
            "--image-folder",
            str(DATA_ROOT),
            "--output-dir",
            str(sample_output_dir),
            "--sample-index",
            str(FIXED_SAMPLE_INDEX),
            "--decoding-strategy",
            "fixed",
            "--fixed-lvr-steps",
            str(FIXED_LVR_STEPS),
            "--attention",
            ATTENTION,
        ],
        cwd=REPO_DIR,
    )

    prediction = json.loads(
        (sample_output_dir / "predictions.jsonl").read_text(encoding="utf-8").splitlines()[0]
    )
    image_path = Path(record["initial_image"])
    if not image_path.is_absolute():
        image_path = DATA_ROOT / image_path
    display(Image(filename=str(image_path)))
    print(f"Split/index: {FIXED_SAMPLE_SPLIT}/{FIXED_SAMPLE_INDEX}")
    print(f"Sample id: {record['id']}")
    print("Expected actions:", " ".join(prediction["expected_actions"]))
    print("Predicted actions:", " ".join(prediction["predicted_actions"]) or "<none>")
    print(f"Latent exit: {prediction['latent_exit_reason']} after {prediction['latent_steps']} steps")
    print(f"Transition token: {prediction['transition_token']!r}")
    print(f"Valid format: {prediction['valid_format']}")
    print(f"Exact match: {prediction['exact_match']}")
    print(f"Reached goal: {prediction['goal_success']} ({prediction['terminal_reason']})")
    print("Action-side model output:", prediction["action_output"])
    print(f"Saved prediction to: {sample_output_dir}")

## Expected data invariants

A successful dataset validation reports:

- 4,000 total samples: 3,600 train, 200 validation, 200 test
- 12,806 transitions
- 16,806 referenced images
- zero board-layout overlap between splits

A successful smoke test reports `status: ok`, matching latent/target token counts, and finite language, latent, and mode-switch losses.